# 1006 24일차

## 1. RNN을 CNN으로

RNN용 3차원 시계열 `(N, timesteps, 특성)`을 `Reshape` 층으로 4차원으로 접어 `Conv2D`에 넣음 (`Reshape` 층은 day23 §6)

![10분 간격 144개 시점을 12×12로 접은 격자. 가로 한 칸은 10분, 세로 한 칸은 2시간 차이라 3×3 커널은 0~2, 12~14, 24~26번 시점을 한 번에 보고 사이 9칸씩을 건너뛴다. Conv1D의 kernel_size=3은 연속된 0~2번만 본다](assets/fold-12x12.svg)

1. `Reshape((12, 12, 13), input_shape=(144, 13))` : 144칸을 12×12로 접고 특성 13개를 채널로. 원소 1872개 그대로
2. `Conv2D` → `Flatten` → `Dense` : 이미지 CNN과 같은 순서

## 2. Conv1D

커널이 시간 축 한 방향으로만 움직이는 CNN. 시간 순서대로 한 줄로 늘어놓고 연속된 `kernel_size`칸의 특성을 모아 학습함

RNN과 같은 3차원 입력을 reshape 없이 그대로 받고, 연속된 칸만 봐서 시간 순서가 섞이지 않음

![Conv2D는 3×3 커널이 6×6 이미지 위를 가로·세로로 움직이며 채널 전체를 봐서 (4, 4, filters)를 만들고, Conv1D는 시간 8칸 × 특성 표에서 연속된 3칸의 특성 전체를 커널로 보며 시간 축으로만 움직여 (6, filters)를 만든다](assets/conv1d-vs-conv2d.svg)

| | Conv2D | Conv1D |
|---|---|---|
| 입력 | `(N, 세로, 가로, 채널)` | `(N, timesteps, 특성)` |
| `kernel_size` | `(n, n)` | `n` |
| 커널이 함께 보는 축 | 채널 전체 | 특성 전체 |
| 출력 (`valid`) | `(N, 세로-n+1, 가로-n+1, filters)` | `(N, timesteps-n+1, filters)` |
| 파라미터 | `filters × (n × n × 채널 + 1)` | `filters × (n × 특성 + 1)` |

- 특성 축이 이미지의 채널 자리. 실제 커널 모양은 `n × 특성`
- 층을 쌓으면 출력 한 칸이 보는 구간이 넓어짐. `kernel_size=3`을 두 층 쌓으면 원래 데이터 5칸
- `kernel_size`는 층 안에서 몇 칸씩 묶어 볼지, timesteps는 데이터를 자를 때 몇 칸씩 묶을지

### 2-1. RNN 모델을 Conv1D로

RNN 층 자리에 `Conv1D`를 넣고, 3차원 출력을 2차원으로 편 뒤 `Dense`로 이어 붙임

1. `Conv1D(filters, kernel_size, input_shape=(timesteps, 특성))` : RNN과 같은 `input_shape`
2. `Flatten` 또는 `GlobalAveragePooling1D` : 3차원 출력을 2차원으로 (day23 §2 cf)
3. `Dense` : 이후는 RNN 모델과 같음

- `GlobalAveragePooling1D`는 시간 축을 평균 내서 `(N, steps, filters)` → `(N, filters)`. GAP(day14 §2)의 1차원 버전
- `kernel_size`를 timesteps와 같은 3으로 두면 출력은 1칸 `(N, 1, filters)`